# Loi de Malus : sans le polariseur

In [ ]:
import glob, os
import ast
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from astropy.io import fits
from scipy.optimize import curve_fit

%matplotlib widget

In [ ]:
def get_colors(n_steps, cmap_name='Blues', vmin=0, vmax=None):
    if vmax is None:
        vmax = n_steps - 1
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = plt.get_cmap(cmap_name)
    return [cmap(norm(i)) for i in range(n_steps)], norm, cmap

def has_key(hdul, key='THETA_R'):
    """Retourne True si THETA_R existe dans le header"""
    return key in hdul[0].header

In [ ]:
#path_save = '/home/lmousset/Projets_Recherche/COSMOCal/Tests_IAS_2026/Data/CosmoCal_data/'
path_save = "C:\\Users\\Administrator\\Documents\\Scripts_Commande_VNA\\CosmoCal_data\\"



# Premier essai mardi 17/02 : quelques points mesurés à vue

In [ ]:
S21 = np.array([-38, -38, -38.7, -39, -39.5, -40, -41.2, -43, -46.5, -52.3, -64.8, -52.4, -47.3, -47, -44.5, -41.2, -41.7, -39])
nmeas = np.size(S21)
angle = np.arange(nmeas)*10

plt.figure()
plt.plot(angle, 10**(S21/10), 'o')

### Première acquisiton mercredi 18/02 : un fichier pour chaque thetaR

In [ ]:
allthetaR = []
allmag = []

path_save2 = path_save + 'Malus_manual/'
os.chdir(path_save2)

i = 0
for file in glob.glob("*thetaR*.fits"):
    print(i)
    print(file)
    hdul = fits.open(path_save2 + file)
    header = hdul[0].header
    thetaR = header['THETA_R']
    allthetaR.append(thetaR)
    mag = hdul[0].data  # Access magnitude
    print(mag.shape)
    allmag.append(mag)
    i+=1

allthetaR = np.array(allthetaR)
allmag = np.array(allmag)
print(allmag.shape)

In [ ]:
ff = 10
colors, norm, cmap = get_colors(ff, cmap_name='viridis', vmin=0, vmax=ff-1)
trace = 0 # S21
plt.figure()
for f in range(ff):
    plt.plot(allthetaR[1:-2], 10**(allmag[1:-2, trace, f]/10), 'o', color=colors[f])

# Nouvelle optimisation : 1 seul fichier

On stop le script pour tourner thetaR 

In [ ]:
# Get data from one file
hdul = fits.open(path_save + "Malus_20260220_103113.fits")
header = hdul[0].header
hdul.info()  # View structure
mag = hdul[0].data  # Access magnitude
mag_lin = 10**(mag/10)

if has_key(hdul, key='THETA_R'):
    thetaR = ast.literal_eval(header['THETA_R'])
else:
    print("THETA_R not found in header, using default values.")
    thetaR = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 
             0, -10, -20, -30, -40, -50, -60, -70, -80, -90, -100, -110, -120] # in degree

print(f"thetaR: {thetaR}")

header

In [ ]:
# Get data from a second file
hdul2 = fits.open(path_save + "Malus_20260220_105809.fits")
header = hdul2[0].header
hdul2.info()  # View structure
mag2 = hdul2[0].data  # Access magnitude
mag2_lin = 10**(mag2/10)

if has_key(hdul2, key='THETA_R'):
    thetaR2 = ast.literal_eval(header['THETA_R'])
else:
    print("THETA_R not found in header, using default values.")
    thetaR2 = [-30, 0, 30] # in degree

print(f"thetaR2: {thetaR2}")

header

In [ ]:
# Plot the mean S21 and S12 for both files
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
axs = axs.ravel()

# S21
axs[0].plot(thetaR, np.mean(mag_lin[:, 0, :], axis=1), 'b+', ms=10, label='File 1')
axs[0].plot(thetaR2, np.mean(mag2_lin[:, 0, :], axis=1), 'r+', ms=10, label='File 2')
axs[0].set_xlabel('thetaR')
axs[0].set_ylabel('S21')
axs[0].legend()

#S12
axs[1].plot(thetaR, np.mean(mag_lin[:, 1, :], axis=1), 'b+', ms=10, label='File 1')
axs[1].plot(thetaR2, np.mean(mag2_lin[:, 1, :], axis=1), 'r+', ms=10, label='File 2')
axs[1].set_xlabel('thetaR')
axs[1].set_ylabel('S12')
axs[1].legend()

axs[0].grid()
axs[1].grid()

fig.tight_layout()
#fig.savefig('../Plots/Malus_20260220_103113et105809_sansPlatine.png', dpi=300)

In [ ]:
# Plot each frequency separately
freqs = [0, 10, 20, 30, 40]
freqs = np.arange(60)
ff = len(freqs)
colors, norm, cmap = get_colors(ff, cmap_name='viridis', vmin=0, vmax=ff-1)

fig, axs = plt.subplots(1, 2, figsize=(12, 6))
axs = axs.ravel()
for f, freq in enumerate(freqs):
    axs[0].plot(thetaR, mag_lin[:, 0, freq], 'o', color=colors[f])
    #axs[0].plot(thetaR2, mag2_lin[:, 0, freq], 'r+', ms=10)
axs[0].set_xlabel('thetaR')
axs[0].set_ylabel('S21')
#axs[0].set_yscale('log')

for f, freq in enumerate(freqs):
    axs[1].plot(thetaR, mag_lin[:, 1, freq], 'o', color=colors[f])
    #axs[1].plot(thetaR2, mag2_lin[:, 1, freq], 'r+', ms=10)
axs[1].set_xlabel('thetaR')
axs[1].set_ylabel('S12')
#axs[1].set_yscale('log')

axs[0].grid()
axs[1].grid()

fig.tight_layout()
#fig.savefig('../Plots/Malus_20260220_103113_all_freqs.png', dpi=300)

In [ ]:

for ang in [-60, -30, 0, 30, 60]:
    idx2 = np.where(np.array(thetaR2) == ang)[0]

    if ang != 0:
        idx = np.where(np.array(allthetaR) == ang)[0]
    else:
        idx = np.where(np.array(allthetaR) == ang)[0][0]
    #print(f"Index of {ang} in allthetaR: {idx}")
    #print(f"Index of {ang} in allthetaR2: {idx2}")

    diff = np.zeros(60)
    for f in range(60):
        #diff[f] = (10**(mag[idx, 0, f]/10) - 10**(mag2[idx2, 0, f]/10))
        diff[f] = (mag[idx, 0, f] - mag2[idx2, 0, f])
    #print(f"Frequency {f}: Difference = {diff}")

    print(f"Angle {ang}°: Mean difference = {np.mean(diff):.2e}, Std difference = {np.std(diff):.2e}")


In [ ]:
#plt.figure()
#plt.plot(diff, 'o')
# plt.axhline(np.mean(diff), linestyle='--', label=f'Mean = {np.mean(diff):.2e}')
# plt.axhline(np.mean(diff)+np.std(diff), linestyle='--')
# plt.axhline(np.mean(diff)-np.std(diff), linestyle='--')
# plt.xlabel('Frequency index')
# plt.ylabel('Difference')
# plt.title(f'Angle = {ang}° - Mean = {np.mean(diff):.2e} - Std = {np.std(diff):.2e}')
# plt.legend()

# Estimation des erreurs

On a fait 5 scans identiques avec alpha = [-90, -90, -45, 0, 45, 90, 90]

In [ ]:
allthetaR = []
allmag = []

os.chdir(path_save)

i = 0
for file in glob.glob("Malus_20260220_14*.fits"):
    #if i>0:
    print(i)
    print(file)
    hdul = fits.open(path_save + file)
    header = hdul[0].header
    thetaR = header['THETA_R']
    print(f"thetaR: {thetaR}")
    allthetaR.append(ast.literal_eval(thetaR))
    mag = hdul[0].data  # Access magnitude
    print(mag.shape)
    allmag.append(mag)
    i+=1

allmag = np.array(allmag)
allthetaR = np.array(allthetaR)
print(allmag.shape)

allmag_lin = 10**(allmag/10)

nscans, nthetaR, nS, nfreqs = allmag_lin.shape

In [ ]:
freqs = [10] # indice de la frequence
colors, norm, cmap = get_colors(nscans, cmap_name='plasma', vmin=0, vmax=nscans-1)

fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs = axs.ravel()

## S21
for m in range(nscans):
    for i, f in enumerate(freqs):
        axs[0].plot(allthetaR[m], allmag_lin[m, :, 0, f], 'o', color=colors[m], label=f'Scan {m}')
axs[0].set_xlabel('thetaR')
axs[0].set_ylabel('S21')
#axs[0].set_yscale('log')
axs[0].grid()
axs[0].legend()

## S12
for m in range(nscans):
    for i, f in enumerate(freqs):
        axs[1].plot(allthetaR[m], allmag_lin[m, :, 1, f], 'o', color=colors[m], label=f'Scan {m}')
axs[1].set_xlabel('thetaR')
axs[1].set_ylabel('S12')
#axs[1].set_yscale('log')
axs[1].grid()
axs[1].legend()

fig.suptitle(f'Frequency index: {freqs[0]}')
fig.tight_layout()

In [ ]:
# Mean and STD sur les 5 scans
mean_over_scans = np.mean(allmag_lin, axis=0)
std_over_scans = np.std(allmag_lin, axis=0) # [nangle, nS, nfreq]

print(mean_over_scans.shape)

In [ ]:
t = 0 # S21 ou S12

fig = plt.figure(figsize=(10, 5))
for a in range(nthetaR): # pour chaque angle
    plt.plot(std_over_scans[a, t, :], label=f'thetaR {allthetaR[0][a]}')
plt.xlabel('Frequency index')
plt.ylabel('STD over scans')
plt.legend()
plt.grid()

#fig.savefig(f"../Plots/Malus_STD_over_scans.png", dpi=300)

In [ ]:
std_over_scans.shape

In [ ]:
t = 0
plt.figure()
plt.plot(allthetaR[0], np.mean(std_over_scans[:, t, :], axis=-1), 'bo')
plt.grid()

In [ ]:
mean_over_scansfreq = np.mean(allmag_lin, axis=(0, 3))
std_over_scansfreq = np.std(allmag_lin, axis=(0, 3)) # [nangle, nS]

print(mean_over_scansfreq.shape)

t = 0
fig = plt.figure()
plt.plot(allthetaR[0], std_over_scansfreq[:, t], 'bo')
plt.xlabel('thetaR')
plt.ylabel('STD over scans and freq')

fig = plt.figure()
plt.plot(mean_over_scansfreq[:, t], std_over_scansfreq[:, t], 'bo')
plt.ylabel('STD over scans and freq')
plt.xlabel('Mean over scans and freq')

In [ ]:
t = 0

m90a = allmag_lin[:,0,t,:]
m90b = allmag_lin[:,1,t,:]

p90a = allmag_lin[:,5,t,:]
p90b = allmag_lin[:,6,t,:]
print(p90b.shape)

m = 2 # indice du scan
plt.figure(figsize=(10, 5))
plt.plot(m90a[m, :], label='m90a')
plt.plot(m90b[m, :], label='m90b')
plt.plot(p90a[m, :], label='p90a')
plt.plot(p90b[m, :], label='p90b')
plt.xlabel('Frequency index')
plt.ylabel('Magnitude')
plt.legend()
plt.title(f'Scan {m}')
plt.grid()

In [ ]:
diff_p90 = p90a - p90b
diff_m90 = m90a - m90b

plt.figure(figsize=(10, 5))
plt.plot(diff_p90[m, :], label='diff_p90')
plt.plot(diff_m90[m, :], label='diff_m90')
plt.xlabel('Frequency index')
plt.title(f'Scan {m}')
plt.grid()
plt.legend()

print(diff_p90.shape)
from astropy.stats import sigma_clipped_stats

mean_p90, median_p90, stddev_p90 = sigma_clipped_stats(diff_p90, sigma=2)
print(mean_p90, median_p90, stddev_p90)

mean_m90, median_m90, stddev_m90 = sigma_clipped_stats(diff_m90, sigma=2)
print(mean_m90, median_m90, stddev_m90)

# Ajustement

In [ ]:
# Get data from one file
hdul = fits.open(path_save + "Malus_20260220_103113.fits")
header = hdul[0].header
hdul.info()  # View structure
mag = hdul[0].data  # Access magnitude
mag_lin = 10**(mag/10)

if has_key(hdul, key='THETA_R'):
    thetaR = ast.literal_eval(header['THETA_R'])
else:
    print("THETA_R not found in header, using default values.")
    thetaR = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 
             0, -10, -20, -30, -40, -50, -60, -70, -80, -90, -100, -110, -120] # in degree

print(f"thetaR: {thetaR}")

header

In [ ]:
# Model function for fitting
def cos2(theta_R, theta_E, A):
    return A * np.cos(np.radians(theta_E - theta_R))**2

def get_chi2(data, model, error):
    chi2 = np.sum((data - model)**2 / error**2)
    return chi2

### Ajustement de la moyenne

In [ ]:
t = 0 # trace index (S21 or S12)
tt = np.arange(-150, 150, 1)

signal = np.mean(mag_lin[:, 0, :], axis=1)
sigma = 1e-6
#sigma = std_over_scans[:, t, f] / np.sqrt(nscans)  # Erreur sur la moyenne

# Fit the data
popt, pcov = curve_fit(cos2, thetaR, signal, sigma=sigma, absolute_sigma=True)
error = np.sqrt(np.diag(pcov))

print("Paramètres optimisés :", popt)
print("Erreurs sur les paramètres :", error)

# Residus et chi2
residuals = signal - cos2(thetaR, *popt)
std_res = np.std(residuals)
print("Écart-type des résidus :", std_res)
chi2 = get_chi2(signal, cos2(thetaR, *popt), std_res)
print("Chi2 :", chi2)

chi2_red = chi2 / (len(thetaR) - len(popt))
print("Chi2 réduit :", chi2_red)

fig = plt.figure()
plt.errorbar(thetaR, signal, yerr=sigma, marker='.', ls='None')
plt.plot(tt, cos2(tt, *popt), 'orange', label='Fit cos^2')
plt.xlabel("thetaR (°)")
plt.ylabel("S21 mean over scans")
plt.title(r'$\theta_E$ = {:.3f} ± {:.3f}°'.format(popt[0], error[0]))
plt.grid(10)

fig.tight_layout()
#fig.savefig(f"../Plots/Malus_Fit_avec_sigma1e-6_moyenne.png", dpi=300)

### Ajustement par fréquence

In [ ]:
# Loop over frequencies
t = 0 # trace index (S21 or S12)

allthetaE = []
allthetaE_err = []

for f in range(60):
    signal = mag_lin[:, 0, f]
    sigma = 1e-6
    #sigma = std_over_scans[:, t, f]

    # Fit the data
    popt, pcov = curve_fit(cos2, thetaR, signal, sigma=sigma, absolute_sigma=True)
    error = np.sqrt(np.diag(pcov))

    print("Paramètres optimisés :", popt)
    allthetaE.append(popt[0])
    print("Erreurs sur les paramètres :", error)
    allthetaE_err.append(error[0])

    # Residus et chi2
    residuals = signal - cos2(thetaR, *popt)
    std_res = np.std(residuals)
    print("Écart-type des résidus :", std_res)
    chi2 = get_chi2(signal, cos2(thetaR, *popt), std_res)
    print("Chi2 :", chi2)

    chi2_red = chi2 / (len(thetaR) - len(popt))
    print("Chi2 réduit :", chi2_red)

    if f == 30:
        fig = plt.figure()
        plt.errorbar(thetaR, signal, yerr=None, marker='.', ls='None')
        plt.plot(tt, cos2(tt, *popt), 'orange', label='Fit cos^2')
        plt.xlabel("thetaR (°)")
        plt.ylabel(f"S21 fréquence {f}")
        plt.title(r'$\theta_E$ = {:.3f} ± {:.3f}°'.format(popt[0], error[0]))
        plt.grid(10)
        fig.tight_layout()

In [ ]:
nfreq = 60

allthetaE = np.array(allthetaE)
allthetaE_err = np.array(allthetaE_err)

mthetaE = np.mean(allthetaE)
print("ThetaE moyen :", mthetaE)
stdthetaE = np.std(allthetaE) / np.sqrt(nfreq)
print("Écart-type de thetaE :", stdthetaE)

# Erreur sur la moyenne de thetaE
erreur = np.sqrt(np.sum(allthetaE_err**2 )) / nfreq
print('Erreur sur thetaE', erreur)

fig = plt.figure()
plt.errorbar(np.arange(len(allthetaE)), allthetaE, yerr=allthetaE_err, fmt='.')
plt.axhline(mthetaE, color='orange', label=r'$Mean$')
plt.axhline(mthetaE-stdthetaE, color='orange', ls='--')
plt.axhline(mthetaE+stdthetaE, color='orange', label=r'$Mean \pm STD/\sqrt{N}$', ls='--')
plt.xlabel("Fréquence index")
plt.ylabel(r'$\theta_E$ (°)')
plt.title(r'$\theta_E$ = {:.3f} ± {:.3f}°'.format(mthetaE, erreur))
plt.grid(10)
plt.legend()
fig.savefig(f"../Plots/Malus_Fit_avec_sigma1e-6_allthetaE", dpi=300)